# 

In [54]:
import pandas as pd
import ast

## Download genome assembly and annotation data for species with representative HGT-chimeras

In [212]:
df=pd.read_csv("/net/bos-nfsisilon/ifs/rc_labs/extavour_lab/rkapoor/hgt_pipeline_redo_2025/SI tables/SI_table_2_v7.tsv",sep="\t",index_col=0)

In [213]:
df=df[df.representative]

In [105]:
import subprocess

def download_ncbi_genome(accession):
    """
    Download and unzip genome data from NCBI Datasets API
    using curl and unzip via subprocess.
    """

    url = (
        f"https://api.ncbi.nlm.nih.gov/datasets/v2alpha/genome/accession/"
        f"{accession}/download"
        f"?include_annotation_type=GENOME_GFF,PROT_FASTA,GENOME_FASTA"
        f"&filename={accession}.zip"
    )

    zip_file = f"{accession}.zip"

    # Run curl
    subprocess.run(
        [
            "curl",
            "-OJX", "GET",
            url,
            "-H", "Accept: application/zip"
        ],
        check=True
    )

    # Run unzip (overwrite like -o)
    subprocess.run(
        ["unzip", "-o", zip_file],
        check=True
    )
    subprocess.run(
        ["rm", zip_file],
        check=True
    )

    return zip_file
for x in set(df.genome):
     download_ncbi_genome(accession)

## Search for metazoan neighbors  +/-3 genes away from HGT-chimeras

In [58]:
from Bio import SeqIO

In [19]:
gmap={}

"""
For each HGT-chimera, write proteins of the three closest genes one either side of the HGT chimera (if available)
store the gene-->neighbor map as dictionary gmap
"""
f=open('neighbor_queries.fasta','w')
for n, dfi in df[df.representative==True].groupby('genome'):
    prot=set(dfi.index)
    fasta_dict = SeqIO.to_dict(SeqIO.parse(f"ncbi_dataset/data/{n}/protein.faa", "fasta"))
    gff=pd.read_csv(f"ncbi_dataset/data/{n}/genomic.gff",sep="\t",comment="#",header=None)
    gene_prot={x.split("gene=")[1].split(";")[0]:x.split(";")[0].split("-")[1] for x in gff[gff[2]=='CDS'].iloc[:,8] if 'gene' in x}
    print(len({x for x in gff[gff[2]=='CDS'].iloc[:,8] if 'gene=' not in x}))
    for p in prot:
        loc=gff[gff[8].str.contains(p)].iloc[0,0]
        gene=gff[gff[8].str.contains(p)].iloc[0,8].split(";gene=")[1].split(";")[0]
        gffi=gff[gff[0]==loc]
        gffi=gffi.sort_values(3)
        gffi=gffi[gffi[2]=='gene']
        gffi=gffi[gffi[8].str.contains('protein_coding')]
        print("gffi",gffi.shape[0])
        gffi.index=list(range(gffi.shape[0]))
        i=gffi[gffi[8].str.contains(gene)].index[0]
      
        genes=set()
        for x in [-3,-2,-1,1,2,3]:
            if i+x in gffi.index:
                g=gffi.loc[i+x,8].split(";gene=")[1].split(";")[0]
                pg=gene_prot[g]
                f.write(f">{pg};{g}"+"\n")
                s=str(fasta_dict[pg].seq)
                f.write(s+"\n")
                genes.add(g)
        gmap[gene]=genes
f.close()

0
gffi 18
gffi 30
gffi 12
gffi 13
gffi 16
gffi 12
gffi 23
gffi 12
13
gffi 47
0
gffi 21
0
gffi 99
gffi 290
0
gffi 5454
0
gffi 1139
gffi 340
gffi 62
gffi 135
gffi 2592
gffi 1139
gffi 166
gffi 1159
gffi 1139
gffi 1139
gffi 2436
0
gffi 10943
0
gffi 62
0
gffi 10
gffi 54
0
gffi 3119
0
gffi 4075
0
gffi 1901
gffi 1495
gffi 1237
gffi 1901
0
gffi 3739
0
gffi 47
0
gffi 50
0
gffi 436
0
gffi 954
0
gffi 1711
0
gffi 134
0
gffi 991
0
gffi 602
gffi 274
gffi 395
gffi 362
0
gffi 443
0
gffi 532
0
gffi 1381
0
gffi 5435
0
gffi 2254
gffi 1598
0
gffi 182
0
gffi 864
0
gffi 1113
0
gffi 1827
0
gffi 945
gffi 2101
gffi 1428
0
gffi 1791
0
gffi 869
gffi 2110
0
gffi 1477
0
gffi 82
0
gffi 4174
0
gffi 2328
0
gffi 1227
0
gffi 1148
gffi 696
gffi 696
gffi 239
gffi 1408
gffi 1408
gffi 1955
gffi 696
0
gffi 2073
0
gffi 536
0
gffi 978
0
gffi 755
0
gffi 401
gffi 425
gffi 637
0
gffi 816
gffi 2
0
gffi 5115
0
gffi 4056
0
gffi 1238
0
gffi 2495
0
gffi 3263
gffi 305
0
gffi 1768
0
gffi 763
0
gffi 381
0
gffi 791
0
gffi 1632
gffi 1207


In [40]:
import pickle
with open('gene_neighbor.pickle', 'wb') as handle:
  pickle.dump(gmap, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [59]:
import pickle
with open('gene_neighbor.pickle', 'rb') as handle:
  gmap=pickle.load(handle)

In [20]:
!sbatch run_diamond_neighbors.sh

Submitted batch job 65023734


In [69]:
%%bash

input_file="neighbor_diamond"
output_dir="neighbor_diamond_split"
mkdir -p "$output_dir"

# AWK script to split the diamond output TSV file
awk -F'\t' -v output_dir="$output_dir" '{
    # Extract the value of the first column
    value = $1

    # Construct the output file path
    output_file = output_dir "/" value ".tsv"

    # Append the current line to the corresponding output file
    print >> output_file
}' "$input_file"

echo "Splitting complete. Split files are stored in $output_dir directory."

Splitting complete. Split files are stored in neighbor_diamond_split directory.


In [214]:
metazoan_genes=set()
a="qseqid sseqid stitle staxids sscinames sphylums skingdoms pident length mismatch gapopen qstart qend sstart send evalue bitscore"
a=a.split(" ")
combined_data=pd.DataFrame(columns=a)
ind=0
import os

for index in os.listdir("neighbor_diamond_split"):
    if '.tsv' in index:
        dfd=pd.read_csv(f"neighbor_diamond_split/{index}",sep="\t",header=None)
        if dfd.iloc[0,6]=='Metazoa':
            metazoan_genes.add(index.split(".tsv")[0]) 
        combined_data.loc[ind,a]=list(dfd.iloc[0,:])
        ind+=1

In [215]:
combined_data.to_csv('neighbor_diamond_search.tsv',sep='\t')

In [216]:
from Bio import SeqIO
metazoan_neighbors={}
for index, row in df.iterrows():
    n=row.genome
    fasta_dict = SeqIO.to_dict(SeqIO.parse(f"ncbi_dataset/data/{n}/protein.faa", "fasta"))
    neighbors=[]
    for x in metazoan_genes:
        if x.split(";")[0] in fasta_dict and x.split(";")[1] in gmap[row.gene]:
            neighbors.append(x)
    metazoan_neighbors[row.gene]=neighbors
    df.loc[index,'metazoan_neighbors']=str(neighbors)
    df.loc[index,'n_metazoan_neighbors']=len(neighbors)

In [217]:
df[df.n_metazoan_neighbors>0].shape[0]

97

## Prepare for long read analysis by making BED files from gff files

In [22]:
def cds_nt_interval_to_genomic_blocks(exons, nt_start, nt_end, sign, contig):
    """
    exons: list of (gff_start, gff_end) with start<=end, ordered in transcript order
    nt_start/nt_end: 1-based inclusive positions on concatenated CDS in transcript order
    returns list of (contig, gff_start, gff_end) tuples (1-based inclusive)
    """
    out = []
    pos = 1  # 1-based start position of current exon in concatenated CDS

    for (gff_start, gff_end) in exons:
        seg_len = gff_end - gff_start + 1
        seg_pos_start = pos
        seg_pos_end = pos + gff_end - gff_start

        # overlap in CDS-nt space
        ov_start = max(nt_start, seg_pos_start)
        ov_end = min(nt_end, seg_pos_end)
        
        if ov_start <= ov_end:
            off0 = ov_start - seg_pos_start  # 0-based offset within exon
            off1 = ov_end - seg_pos_start

            if sign == "+":
                g_start = gff_start + off0
                g_end = gff_start + off1
            else:
                # transcript moves from gff_end downwards on '-' strand
                g_end = gff_end - off0
                g_start = gff_end - off1
                if g_start > g_end:
                    g_start, g_end = g_end, g_start

            ##bed start is shifted -1 (0-index)
            out.append((contig, int(g_start)-1, int(g_end)))

        pos += seg_len
        if pos > nt_end:
            break

    return out
import ast
"""
takes a genome accession
for all genome accessions, looks up the chimeras in that genome
writes a bed file in nucleotide coordinates for all hgt and metazoan intervals in the chimera
"""
def write_chimera_intervals_bed(genome):
    f=open(f'ncbi_dataset/data/{genome}/chimera_intervals.bed','w')
    
    gff=pd.read_csv(f"ncbi_dataset/data/{genome}/genomic.gff",sep="\t",comment="#",header=None)
    cds=gff[gff[2]=='CDS']
    for gene in df[df.genome==genome].index:
        g=df[df.index==gene]
        gffx=cds[cds[8].str.contains(f"ID=cds-{gene};")]
        sign=gffx.iloc[0,6]
        exons=[]
        ##adjust order of cds exons according to strand
        if sign=='-':
            gffx=gffx.sort_values(3,ascending=False)
        elif sign=='+':
            gffx=gffx.sort_values(3,ascending=True)
        for index, row in gffx.iterrows():
            exons.append((row[3],row[4]))
        sign=gffx.iloc[0,6]
        contig=gffx.iloc[0,0]
        for interval_name in [x for x in g.columns if 'interval' in x]:
            interval=g.loc[:,interval_name].values[0]
            if str(interval)!='nan':
                interval=ast.literal_eval(interval)
                start,stop=interval
                cds_start=(start-1)*3+1
                cds_end=stop*3
                output=cds_nt_interval_to_genomic_blocks(exons, cds_start, cds_end,sign,contig)
                n=f"{gene}_{interval_name}_{interval}".replace(" ","")
                for o in output:
                    a="\t".join(list([str(x) for x in list(o)]))+"\t"+n+"\n"
                    f.write(a)
    f.close()
    

In [23]:
##writes a bed file for all genes in a genome
def write_gene_bed_file(genome):
    gff=pd.read_csv(f"ncbi_dataset/data/{genome}/genomic.gff",sep="\t",comment="#", header=None)
    gff=gff[gff[2]=='gene']
    gff[9]=[x.split("ID=gene-")[1].split(";")[0] for x in gff[8]]
    gff=gff.loc[:,[0,3,4,9]]
    gff=gff.set_index(0)
    gff[3]=gff[3]-1
    gff.to_csv(f"ncbi_dataset/data/{genome}/genes.bed", sep="\t", header=False)

In [27]:
## combined_bed_files
def combine_beds(genome):
    gene_bed = f"ncbi_dataset/data/{genome}/genes.bed"
    chimera_bed = f"ncbi_dataset/data/{genome}/chimera_intervals.bed"
    concatenated_bed = f"ncbi_dataset/data/{genome}/concatenated.bed"

    with open(concatenated_bed, "w") as outfile:
        for fname in [gene_bed, chimera_bed]:
            with open(fname) as infile:
                outfile.write(infile.read())

In [30]:
## for each genome, write a bed file for chimera intervals, bed file for all genes, then combine the beds
for genome in set(df.genome):
    write_chimera_intervals_bed(genome)
    write_gene_bed_file(genome)
    combine_beds(genome)

## Run long read analysis

In [12]:
import pandas as pd
srr=pd.read_csv("srr_list.txt",sep="\t")
srr=srr[srr.run.astype(str)!='nan']
srr['run']=[x.replace("\t","") for x in srr.run]
srr['type']=[x.replace(" ","") for x in srr['type']]

import os
for index, row in srr.iterrows():
  
    g, t,r=row.genome, row['type'], row.run
    if r not in os.listdir("fastq_mapping") or 'overlaps.txt' not in os.listdir(f"fastq_mapping/{r}"):
        r=r.strip()
 
        !sbatch requeue_run_ONT_mapper.sh "$r" "$g" "$t"

In [14]:
##timed out after 2 days, excluded from final analysis 
exclude=[]
import os
for x in srr.run:
    if 'overlaps.txt' not in os.listdir(f"fastq_mapping/{x}"):
        exclude.append(x)
        

In [15]:
srr[srr.run.isin(exclude)]

,genome,species,type,run
30,GCF_950023065.1,Planococcus citri,map-pb,ERR10906111
34,GCF_023897955.1,Schistocerca gregaria,map-hifi,SRR17000761
49,GCF_023864275.1,Schistocerca cancellata,map-hifi,SRR18788529


In [16]:
srr[~srr['run'].isin(exclude)].to_csv("srr_runs_final.tsv",sep="\t")

In [19]:
srr=pd.read_csv("srr_runs_final.tsv",sep="\t",index_col=0)


In [66]:
## create a dataframe with overlap data for all hgt-chimera genes, intervals and ther metazoan neighbors with raw reads
import gc
df_result=pd.DataFrame()
for index, row in srr.iterrows():
    try:
        dfo=pd.read_csv(f"fastq_mapping/{row.run}/overlaps.txt",sep="\t",header=None,dtype=str)
        genome=row.genome
        species=row.species
        chimeras=set(df[df.genome==genome].gene)
        neighbors=set()
        for c in chimeras:
            neighbors=neighbors|set([x.split(";")[1] for x in metazoan_neighbors[c]])
        dfoc=dfo[(dfo[9].str.contains('XP_'))|(dfo[9].isin(chimeras))|(dfo[9].isin(neighbors))]
        dfoc.loc[:,['species','genome']]=species,genome
        df_result=pd.concat([df_result,dfoc],axis=0)

        del dfo
        gc.collect()
        
    
    except:
        print(row)
    

genome         GCF_014529535.1
species    Bradysia coprophila
type                   map-ont
run                 SRR9971648
Name: 77, dtype: object


In [72]:
df_result=df_result.drop(6,axis=1)
df_result=df_result.drop(5,axis=1)
df_result=df_result.drop(4,axis=1)

In [77]:

df_result.columns=['scaffold','read_start','read_end','read_name','feature_start','feature_end','feature_name','species','genome']

In [92]:
##total number of HGT-chimeras tested for long read evidence
df[df.genome.isin(srr.genome)].shape[0]

87

In [182]:
## collect hgt-chimeras with evidence of long reads overlapping both hgt and metazoan intervals
hgt_overlaps=set([x.split("_HGT")[0] for x in df_result[df_result.feature_name.str.contains('HGT_')].feature_name])
overlaps_hgt_meta=set()
for x in hgt_overlaps:
    meta_reads=set(df_result[df_result.feature_name.str.contains(x)&df_result.feature_name.str.contains('Metazoan_')].read_name)
    hgt_reads=set(df_result[df_result.feature_name.str.contains(x)&df_result.feature_name.str.contains('HGT_')].read_name)
    if len(meta_reads&hgt_reads)>0:
        overlaps_hgt_meta.add(x)
len(overlaps_hgt_meta)

73

In [183]:
## collect hgt-chimeras with evidence of long reads overlapping neighboring genes
overlaps_neighbors=set()
for x in set(df[df.genome.isin(srr.genome)].gene):
    chimera_reads=set(df_result[df_result.feature_name.str.contains(x)].read_name)
    neighbors=[x.split(";")[1] for x in metazoan_neighbors[x]]
    neighbor_reads=set(df_result[df_result.feature_name.isin(neighbors)].read_name)
    if len(chimera_reads&neighbor_reads)>0:
        overlaps_neighbors.add(x)
len(overlaps_neighbors)

57

In [104]:
df_result.to_csv("combined_long_read_analysis_data.tsv",sep="\t")

In [145]:
!mkdir 'contamination_analysis_data'

In [146]:
!mv neighbor_diamond_search.tsv contamination_analysis_data/neighbor_diamond_search.tsv

In [148]:
!mv srr_runs_final.tsv contamination_analysis_data/srr_runs_final.tsv

In [241]:
!rm -r contamination_analysis_data/.ipynb_checkpoints/

In [242]:
!tar -zcvf contamination_analysis_data.tar.gz contamination_analysis_data

contamination_analysis_data/
contamination_analysis_data/combined_long_read_analysis_data.tsv
contamination_analysis_data/metazoan_neighbors.tsv
contamination_analysis_data/neighbor_diamond_search.tsv
contamination_analysis_data/srr_runs_final.tsv


In [220]:
for index,row in df.iterrows():
    if row.gene in overlaps_neighbors:
        df.loc[index,'read_spans_neighbors']=True
    elif row.genome in set(srr.genome):
        df.loc[index,'read_spans_neighbors']=False
    if index in overlaps_hgt_meta:
        df.loc[index,'read_spans_hgt/meta']=True
    elif row.genome in set(srr.genome):
        df.loc[index,'read_spans_hgt/meta']=False

In [197]:
df=df.loc[:,['gene','cluster', 'species','metazoan_neighbors', 'n_metazoan_neighbors',
       'read_spans_neighbors', 'read_spans_hgt/meta']]

In [223]:
df[df.read_spans_neighbors==True].shape[0]

57

In [224]:
df[df["read_spans_hgt/meta"]==True].shape[0]

73

In [237]:
##species-specific chimeras supported by long reads
df[(df.cluster>23)&(df['read_spans_hgt/meta']==True)].shape[0]

54

In [238]:
##species-specific chimeras supported by long reads
df[(df.cluster>23)&(df['read_spans_hgt/meta'].astype(str)!='nan')].shape[0]

65

In [225]:
df.to_csv("contamination_data.tsv",sep="\t")

In [229]:
df.loc[:,'metazoan_neighbors'].to_csv('contamination_analysis_data/metazoan_neighbors.tsv',sep='\t')